In [1]:
!pip install datasets
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 19.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 1.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
from transformers import AutoModel
from datasets import load_dataset
from transformers import AutoTokenizer
import random
import math
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from sacrebleu import corpus_bleu
from torch.utils.data.distributed import DistributedSampler
from transformers import AutoModelForSeq2SeqLM
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import get_linear_schedule_with_warmup
torch.manual_seed(66)

Mounted at /content/drive


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [4]:
train_dataset = load_dataset("squad", split="train")

test_dataset = load_dataset("squad", split="validation")

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [5]:
print(train_dataset)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})


In [6]:
print(train_dataset['id'][0])
print(train_dataset['title'][0])
print(train_dataset['context'][0])
print(train_dataset['question'][0])
print(train_dataset['answers'][0])

5733be284776f41900661182
University_of_Notre_Dame
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [7]:
print(test_dataset['id'][0])
print(test_dataset['title'][0])
print(test_dataset['context'][0])
print(test_dataset['question'][0])
print(test_dataset['answers'][0])

56be4db0acb8001400a502ec
Super_Bowl_50
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.
Which NFL team represented the AFC at Super Bowl 50?
{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [177, 177, 177]}


In [8]:
print(test_dataset['id'][2])
print(test_dataset['title'][2])
print(test_dataset['context'][2])
print(test_dataset['question'][2])
print(test_dataset['answers'][2])

56be4db0acb8001400a502ee
Super_Bowl_50
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.
Where did Super Bowl 50 take place?
{'text': ['Santa Clara, California', "Levi's Stadium", "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."], 'answer_start': [

In [9]:
model_name = 'facebook/bart-base'

#Data Tokenize and Preprocessing

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name, model_max_length = 512)

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
# single_item = train_dataset[0]
# print(single_item)

# context = single_item['context'].strip()
# question = single_item['question'].strip()
# answer_text = single_item['answers']['text'][0].strip()
# question_tokens = tokenizer.tokenize(question)
# input_ids = tokenizer(question)['input_ids']

# answer_start = single_item['answers']['answer_start'][0]
# answer_end = answer_start + len(answer_text) - 1

# print(single_item['question'])
# print(question)
# print(question_tokens)
# print(input_ids)
# print(answer_text)
# print(context[answer_end])

In [12]:
def tokenize_single_sample(dict_row):
  context = dict_row['context'].strip()
  context_tokens = tokenizer.tokenize(context, truncation=True, max_length=512)

  question = dict_row['question'].strip()
  question_tokens = tokenizer.tokenize(question, truncation=True, max_length=512)
  question_input_ids = tokenizer(question, truncation=True, max_length=512)['input_ids']
  quesiton_input_positions = torch.tensor(list(range(len(question_input_ids))))

  answer_text = dict_row['answers']['text'][0].strip()
  answer_tokens = tokenizer.tokenize(answer_text, truncation=True, max_length=512)
  answer_start = dict_row['answers']['answer_start'][0]
  answer_end = answer_start + len(answer_text) - 1
  # answer token start and end position at the context token array
  answer_token_start = len(tokenizer(context[:answer_start], truncation=True, max_length=512)['input_ids']) - 1
  answer_token_end = len(tokenizer(context[:answer_end], truncation=True, max_length=512)['input_ids']) - 1


  # Section-1: Question Generation (Input: will be Context and Answer text, Output: Question)
  tokenized_input_qg = tokenizer(context +'</s><s>'+ answer_text, truncation=True, max_length=512)['input_ids']
  # task=0 for Question Generation, 1 for Question Answering and 2 for Uncommon Word generation
  task = 0
  tokenized_input_position_qg = torch.tensor(list(range(len(context_tokens)+2)) + list(range(len(answer_tokens)+2)))
  task_embedding_input_qg = torch.tensor([task] * len(tokenized_input_qg))
  # Segment ID = 0 for context, 1 for answer and 2 for question. and +2 is for token <s> and </s> for each segment
  segment_embedding_input_qg = torch.tensor([0] * (len(context_tokens)+2) + [1] * (len(answer_tokens)+2))
  #The following code I am writing for just patching in error handling
  if segment_embedding_input_qg.size(0) > 512:
    desired_length_a = int(0.95 * len(tokenized_input_qg))
    desired_length_b = len(tokenized_input_qg) - desired_length_a
    segment_embedding_input_qg = torch.tensor([0] * desired_length_a  + [1] * desired_length_b)

  # Section-2: Answering Questions (Input: Context and Question, Output: Answer )
  tokenized_input_qa = tokenizer(context + '</s><s>' +question, truncation=True, max_length=512)['input_ids']
  # task=0 for Question Generation, 1 for Question Answering and 2 for Uncommon Word generation
  task = 1
  tokenized_input_position_qa = torch.tensor(list(range(len(context_tokens)+2)) + list(range(len(question_tokens)+2)))
  task_embedding_input_qa = torch.tensor([task] * len(tokenized_input_qa))
  # Segment ID = 0 for context, 1 for answer and 2 for question. and +2 is for token <s> and </s> for each segment
  segment_embedding_input_qa = torch.tensor([0] * (len(context_tokens)+2) + [2] * (len(question_tokens)+2))
  #The following code I am writing for just patching in error handling
  if segment_embedding_input_qa.size(0) > 512:
    desired_length_a = int(0.90 * len(tokenized_input_qa))
    desired_length_b = len(tokenized_input_qa) - desired_length_a
    segment_embedding_input_qa = torch.tensor([0] * desired_length_a  + [1] * desired_length_b)

  # Section-3: Uncommon Word Generation (Input: Context, Output: Word Distribution)
  tokenized_input_uw = tokenizer(context, truncation=True, max_length=512)['input_ids']
  # task=0 for Question Generation, 1 for Question Answering and 2 for Uncommon Word generation
  task = 2
  tokenized_input_position_uw = torch.tensor(list(range(len(context_tokens)+2)))
  task_embedding_input_uw = torch.tensor([task] * len(tokenized_input_uw))
  # Segment ID = 0 for context, 1 for answer and 2 for question. and +2 is for token <s> and </s> for each segment
  segment_embedding_input_uw = torch.tensor([0] * (len(context_tokens)+2))
  if segment_embedding_input_uw.size(0) > 512:
    segment_embedding_input_uw = torch.tensor([0] * len(tokenized_input_uw))

  return {
      'context_tokens': context_tokens,
      'question_tokens': question_tokens,
      'question_input_ids': question_input_ids,
      'quesiton_input_positions': quesiton_input_positions,
      'answer_tokens': answer_tokens,
      'answer_token_start': answer_token_start,
      'answer_token_end': answer_token_end,
      'tokenized_input_qg': tokenized_input_qg,
      'tokenized_input_position_qg': tokenized_input_position_qg,
      'task_embedding_input_qg': task_embedding_input_qg,
      'segment_embedding_input_qg': segment_embedding_input_qg,
      'tokenized_input_qa': tokenized_input_qa,
      'tokenized_input_position_qa': tokenized_input_position_qa,
      'task_embedding_input_qa': task_embedding_input_qa,
      'segment_embedding_input_qa': segment_embedding_input_qa,
      'tokenized_input_uw': tokenized_input_uw,
      'tokenized_input_position_uw': tokenized_input_position_uw,
      'task_embedding_input_uw': task_embedding_input_uw,
      'segment_embedding_input_uw': segment_embedding_input_uw
  }


In [13]:
train_dataset = train_dataset.map(tokenize_single_sample)
test_dataset = test_dataset.map(tokenize_single_sample)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [14]:
print(train_dataset['question_tokens'][0])
print(train_dataset['question_input_ids'][0])

['To', 'Ġwhom', 'Ġdid', 'Ġthe', 'ĠVirgin', 'ĠMary', 'Ġallegedly', 'Ġappear', 'Ġin', 'Ġ18', '58', 'Ġin', 'ĠL', 'our', 'des', 'ĠFrance', '?']
[0, 3972, 2661, 222, 5, 9880, 2708, 2346, 2082, 11, 504, 4432, 11, 226, 2126, 10067, 1470, 116, 2]


#Model Definition

In [15]:
class EmbeddingLayer(nn.Module):
  def __init__(self, embedding_dim):
    super(EmbeddingLayer, self).__init__()
    self.embedding_dim = embedding_dim
    self.bart_embedding = AutoModel.from_pretrained(model_name).shared
    # #Converting BART embedding to desired dimension
    # self.word_embedding = nn.Linear(self.bart_embedding.weight.shape[1], embedding_dim)
    #position embedding is done combined with word embedding in bart
    # self.positional_embedding = AutoModel.from_pretrained(model_name).get_input_embeddings().position_embeddings
    self.task_embedding = nn.Embedding(3, embedding_dim)
    self.segment_embedding = nn.Embedding(3, embedding_dim)


  def forward(self, word_input, position_input, task_input, segment_input):
    # print(word_input.shape)
    word_input = self.bart_embedding(word_input)
    # # print(word_input.shape)
    # word_input = self.word_embedding(word_input)
    # print(word_input.shape)
    # position_input = self.positional_embedding(position_input)
    task_input = self.task_embedding(task_input)
    # print(task_input.shape)
    segment_input = self.segment_embedding(segment_input)
    # print(segment_input.shape)

    embedding_output = word_input + (task_input + segment_input) / math.sqrt(self.embedding_dim)
    return embedding_output

In [16]:
def create_distillation_mask(input_tokens):
  num_rows, num_cols = input_tokens.shape
  percent_true = 0.10
  num_cols -= 2 #subtracting 2 as input id contains <s> and </s>
  # Number of True elements per row
  num_true_per_row = int(num_cols * percent_true)

  # Create and shuffle each row, then stack them into a 2D tensor
  rows = [torch.cat((torch.ones(num_true_per_row, dtype=torch.bool),
                    torch.zeros(num_cols - num_true_per_row, dtype=torch.bool)))[torch.randperm(num_cols)]
          for _ in range(num_rows)]
  b = torch.stack(rows)

  # Add False columns at the beginning and end of each row for <s> and </s>
  b = torch.cat((torch.zeros(num_rows, 1, dtype=torch.bool), b, torch.zeros(num_rows, 1, dtype=torch.bool)), dim=1)

  return b

In [17]:
class QuesitonGenerationWithKnowledgeDist(nn.Module):
  def __init__(self, embedding_dim, vocab_size, pad_token_id):
    super(QuesitonGenerationWithKnowledgeDist, self).__init__()
    self.embedding_dim = embedding_dim
    self.vocab_size = vocab_size
    self.pad_token_id = pad_token_id

    self.embedding_layer = EmbeddingLayer(embedding_dim = embedding_dim)
    self.primal_dual_encoder = AutoModel.from_pretrained(model_name).encoder
    self.question_decoder = AutoModel.from_pretrained(model_name).decoder
    self.output_layer_qg = AutoModelForSeq2SeqLM.from_pretrained(model_name).lm_head
    self.start_index_ff_qa = nn.Linear(self.embedding_dim, 1)
    self.end_index_ff_qa = nn.Linear(self.embedding_dim*2, 1)
    self.softmax = nn.Softmax(dim = -1)

    self.pretrained_model_no_grad_uw = AutoModel.from_pretrained(model_name)
    self.Wm_uw = nn.Linear(self.embedding_dim, self.vocab_size)



  def forward(self, tokenized_input_qg, tokenized_input_position_qg, task_embedding_input_qg, segment_embedding_input_qg, question_input_ids,
              tokenized_input_qa, tokenized_input_position_qa, task_embedding_input_qa, segment_embedding_input_qa,
              tokenized_input_uw, tokenized_input_position_uw, task_embedding_input_uw, segment_embedding_input_uw):

    ##Question Generation Task
    #At first embed the concatenated passage and answer with respective task and segment
    qg_embedded = self.embedding_layer(tokenized_input_qg, tokenized_input_position_qg, task_embedding_input_qg, segment_embedding_input_qg)
    #creating attention mask / a binary array where 1 will be in the place of data and 0 in the place of padding
    qg_attention_mask = tokenized_input_qg != self.pad_token_id
    # qg_attention_mask = torch.ones_like(tokenized_input_qg)
    # qg_attention_mask[tokenized_input_qg == self.pad_token_id] = 0
    #Input to the primal dual encoder is the embedded input and its attention mask
    qg_encoded_last_hidden_state = self.primal_dual_encoder(inputs_embeds=qg_embedded, attention_mask=qg_attention_mask).last_hidden_state
    # qg_question_id_attention_mask = torch.ones_like(question_input_ids)
    # # print("Question Input =",question_input_ids.shape)
    # qg_question_id_attention_mask[question_input_ids == self.pad_token_id] =
    #Trimming this otherwise the predicted values of question generation seems to produce end of sequence token just after the first word
    trimmed_question_input_ids = question_input_ids[:, :-1]
    #Attention mask for the question
    qg_question_id_attention_mask = trimmed_question_input_ids != self.pad_token_id
    #Decoder inputs are trimmed question along with mask and the last layer of the encoder
    qg_decoded_last_hidden_state = self.question_decoder(input_ids=trimmed_question_input_ids, attention_mask=qg_question_id_attention_mask, encoder_hidden_states=qg_encoded_last_hidden_state).last_hidden_state
    #Passing the generated decoded output to sequence to sequence layer
    qg_generated_questions = self.output_layer_qg(qg_decoded_last_hidden_state)




    ##Question Answering Task
    qa_embedded = self.embedding_layer(tokenized_input_qa, tokenized_input_position_qa, task_embedding_input_qa, segment_embedding_input_qa)
    qa_attention_mask = tokenized_input_qa != self.pad_token_id
    # qa_attention_mask[tokenized_input_qa == self.pad_token_id] = 0
    qa_encoded_last_hidden_state = self.primal_dual_encoder(inputs_embeds=qa_embedded, attention_mask=qa_attention_mask).last_hidden_state
    # print(qa_encoded_last_hidden_state.shape)
    start_index_logits = self.start_index_ff_qa(qa_encoded_last_hidden_state).squeeze(-1)
    # qa_attention_mask = qa_attention_mask.bool()
    # start_index_logits[~qa_attention_mask] = float('-inf')
    start_index_logits_masked = start_index_logits.masked_fill(~qa_attention_mask, float('-inf'))
    start_index_sm = self.softmax(start_index_logits_masked)
    final_start_idx = torch.argmax(start_index_sm, dim=-1, keepdim=True)
    # ignore_for_end_index_mask = (torch.arange(qa_attention_mask.size(1)) <= final_start_idx)  | ~qa_attention_mask
    #The following code is for the places that needs to ignore while deciding the last
    #index of the answer. This is as because, if we the last index should be in between the first index
    #and the starting of the padding index. Therefore masking the space
    temp_tensor = torch.arange(qa_attention_mask.size(1))
    temp_tensor = temp_tensor.to(device)
    ignore_for_end_index_mask = temp_tensor <= final_start_idx
    ignore_for_end_index_mask = ignore_for_end_index_mask | ~qa_attention_mask
    #Repeating the tensor of the layer of final start index
    #To match the shape of the last index of the encoder
    #Because the input for the last index of the answer is the concatenation of
    #last layer of encoder output and the layer of hidden layer to the first input index
    repeated_tensor_of_final_start_index = torch.gather(qa_encoded_last_hidden_state, 1, final_start_idx.unsqueeze(2).expand(-1, qa_encoded_last_hidden_state.size(1), qa_encoded_last_hidden_state.size(2)))
    qa_encoded_last_hidden_state = torch.cat((qa_encoded_last_hidden_state, repeated_tensor_of_final_start_index), dim=-1)
    end_index_logits = self.end_index_ff_qa(qa_encoded_last_hidden_state).squeeze(-1)
    # end_index_logits[ignore_for_end_index_mask] = float('-inf')
    #Masking away all the score till start position and the last paddings
    end_index_logits_masked = end_index_logits.masked_fill(ignore_for_end_index_mask, float('-inf'))
    end_index_sm = self.softmax(end_index_logits_masked)



    ##Uncommon Word Generation
    #Input embedding. It is embedidng the passage
    uw_embedded = self.embedding_layer(tokenized_input_uw, tokenized_input_position_uw, task_embedding_input_uw, segment_embedding_input_uw)
    uw_attention_mask = tokenized_input_uw != self.pad_token_id
    # qa_attention_mask[tokenized_input_qa == self.pad_token_id] = 0
    uw_encoded_last_hidden_state = self.primal_dual_encoder(inputs_embeds=uw_embedded, attention_mask=uw_attention_mask).last_hidden_state
    #Distillation mask is being used for randomly masking some words
    #With a hope of generalization
    uw_distillation_mask = create_distillation_mask(tokenized_input_uw).to(device)
    #Using no gradiant to keep the pretrained model intact
    with torch.no_grad():
      uw_pretrained_encoded_last_hidden_state = self.pretrained_model_no_grad_uw(input_ids=tokenized_input_uw, attention_mask=uw_distillation_mask, decoder_input_ids = tokenized_input_uw).last_hidden_state
    masked_word_encoding_pre = uw_pretrained_encoded_last_hidden_state.masked_select(uw_distillation_mask.unsqueeze(-1)).view(-1, uw_pretrained_encoded_last_hidden_state.shape[-1])
    y_pre = self.softmax(self.Wm_uw(masked_word_encoding_pre))
    # print(masked_word_encoding_pre.shape)
    masked_word_encoding_en = uw_encoded_last_hidden_state.masked_select(uw_distillation_mask.unsqueeze(-1)).view(-1, uw_encoded_last_hidden_state.shape[-1])
    y_en = self.softmax(self.Wm_uw(masked_word_encoding_en))



    return qg_generated_questions, start_index_logits, end_index_logits, start_index_sm, end_index_sm, y_pre, y_en


#Training_Model

##Custom Collate Function

In [18]:
def padding_all_tokens(all_tokens):
    max_length = max(len(token_row) for token_row in all_tokens)
    padded_tokens = []
    for token in all_tokens:
        padded_row = token + ['<pad>'] * (max_length - len(token))
        padded_tokens.append(padded_row)
    return padded_tokens

def padding_all_sequences(all_sequences):
    return pad_sequence(all_sequences, batch_first=True, padding_value=tokenizer.pad_token_id)

In [19]:
def custom_collate(batch):
  context_tokens = padding_all_tokens([item['context_tokens'] for item in batch])

  question_tokens = padding_all_tokens([item['question_tokens'] for item in batch])
  question_input_ids = padding_all_sequences([torch.tensor(item['question_input_ids']) for item in batch])
  quesiton_input_positions = padding_all_sequences([torch.tensor(item['quesiton_input_positions']) for item in batch])

  answer_tokens = padding_all_tokens([item['answer_tokens'] for item in batch])
  answer_token_start = torch.tensor([torch.tensor(item['answer_token_start']) for item in batch])
  answer_token_end = torch.tensor([torch.tensor(item['answer_token_end']) for item in batch])

  tokenized_input_qg = padding_all_sequences([torch.tensor(item['tokenized_input_qg']) for item in batch])
  tokenized_input_position_qg = padding_all_sequences([torch.tensor(item['tokenized_input_position_qg']) for item in batch])
  task_embedding_input_qg = padding_all_sequences([torch.tensor(item['task_embedding_input_qg']) for item in batch])
  segment_embedding_input_qg = padding_all_sequences([torch.tensor(item['segment_embedding_input_qg']) for item in batch])

  tokenized_input_qa = padding_all_sequences([torch.tensor(item['tokenized_input_qa']) for item in batch])
  tokenized_input_position_qa = padding_all_sequences([torch.tensor(item['tokenized_input_position_qa']) for item in batch])
  task_embedding_input_qa = padding_all_sequences([torch.tensor(item['task_embedding_input_qa']) for item in batch])
  segment_embedding_input_qa = padding_all_sequences([torch.tensor(item['segment_embedding_input_qa']) for item in batch])

  tokenized_input_uw = padding_all_sequences([torch.tensor(item['tokenized_input_uw']) for item in batch])
  tokenized_input_position_uw = padding_all_sequences([torch.tensor(item['tokenized_input_position_uw']) for item in batch])
  task_embedding_input_uw = padding_all_sequences([torch.tensor(item['task_embedding_input_uw']) for item in batch])
  segment_embedding_input_uw = padding_all_sequences([torch.tensor(item['segment_embedding_input_uw']) for item in batch])

  return {
      'context_tokens': context_tokens,
      'question_tokens': question_tokens,
      'question_input_ids': question_input_ids,
      'quesiton_input_positions': quesiton_input_positions,
      'answer_tokens': answer_tokens,
      'answer_token_start': answer_token_start,
      'answer_token_end': answer_token_end,
      'tokenized_input_qg': tokenized_input_qg,
      'tokenized_input_position_qg': tokenized_input_position_qg,
      'task_embedding_input_qg': task_embedding_input_qg,
      'segment_embedding_input_qg': segment_embedding_input_qg,
      'tokenized_input_qa': tokenized_input_qa,
      'tokenized_input_position_qa': tokenized_input_position_qa,
      'task_embedding_input_qa': task_embedding_input_qa,
      'segment_embedding_input_qa': segment_embedding_input_qa,
      'tokenized_input_uw': tokenized_input_uw,
      'tokenized_input_position_uw': tokenized_input_position_uw,
      'task_embedding_input_uw': task_embedding_input_uw,
      'segment_embedding_input_uw': segment_embedding_input_uw
  }

#Training_Phase

In [37]:
model_name = 'facebook/bart-base'

dimension_of_model = AutoModel.from_pretrained(model_name).shared.weight.shape[1]
print(dimension_of_model)

batch_size = 4
num_epochs = 2
lr = 3e-5

alpha = 0.8
beta = 0.15

768


In [38]:
store_train_dataset = train_dataset
store_test_dataset = test_dataset

In [39]:
indices = list(range(10000))
train_dataset = store_train_dataset.select(indices)
indices = list(range(1000))
test_dataset = store_test_dataset.select(indices)

In [40]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, collate_fn= custom_collate)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn= custom_collate)

In [41]:
model = QuesitonGenerationWithKnowledgeDist(embedding_dim= dimension_of_model, vocab_size = tokenizer.vocab_size, pad_token_id=tokenizer.pad_token_id)
model = model.to(device)

In [42]:
print(model)

QuesitonGenerationWithKnowledgeDist(
  (embedding_layer): EmbeddingLayer(
    (bart_embedding): Embedding(50265, 768, padding_idx=1)
    (task_embedding): Embedding(3, 768)
    (segment_embedding): Embedding(3, 768)
  )
  (primal_dual_encoder): BartEncoder(
    (embed_tokens): Embedding(50265, 768, padding_idx=1)
    (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
    (layers): ModuleList(
      (0-5): 6 x BartEncoderLayer(
        (self_attn): BartAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(

In [43]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

total_training_steps = len(train_dataloader) * num_epochs
learning_rate_warmup_steps = 50
scheduler = get_linear_schedule_with_warmup(optimizer, learning_rate_warmup_steps, total_training_steps)

best_val_loss = float('inf')
best_model = None

In [44]:
for epoch in range(num_epochs):
  print(f"Epoch {epoch + 1}/{num_epochs}")
  model.train()
  train_loss = 0
  for batch in tqdm(train_dataloader):
    optimizer.zero_grad()
    gen_qs, start_logits, end_logits, start_smax, end_smax, y_pre, y_en = model(batch['tokenized_input_qg'].to(device), batch['tokenized_input_position_qg'].to(device), batch['task_embedding_input_qg'].to(device), batch['segment_embedding_input_qg'].to(device), batch['question_input_ids'].to(device),
                                                                                batch['tokenized_input_qa'].to(device), batch['tokenized_input_position_qa'].to(device), batch['task_embedding_input_qa'].to(device), batch['segment_embedding_input_qa'].to(device),
                                                                                batch['tokenized_input_uw'].to(device), batch['tokenized_input_position_uw'].to(device), batch['task_embedding_input_uw'].to(device), batch['segment_embedding_input_uw'].to(device))
    #Trimming the input ID shape by one to match the shape of input
    #During input, its tail was dropped to save it from producing <\s> by bias
    target_qs = batch['question_input_ids'][:, 1:].to(device)
    qg_loss = criterion(gen_qs.view(-1, gen_qs.shape[-1]), target_qs.view(-1))

    answer_start_pos = batch['answer_token_start'].to(device)
    answer_end_pos = batch['answer_token_end'].to(device)
    # print(criterion(start_logits, answer_start_pos).item(), criterion(end_logits, answer_end_pos).item())
    # print("Start Logits = ",start_logits)
    # print("Start Positions = ",answer_start_pos)
    # print("End Logits = ",end_logits)
    # print("End Positions = ",answer_end_pos)
    qa_loss = criterion(start_logits, answer_start_pos) + criterion(end_logits, answer_end_pos)
    # print(criterion(start_logits, answer_start_pos).item(), criterion(end_logits, answer_end_pos).item())
    uw_loss = -torch.sum(y_en * torch.log(y_pre))
    # print(qg_loss.item(), qa_loss.item(), uw_loss.item())
    loss_in_epoch = qg_loss + alpha * qa_loss + beta * uw_loss
    loss_in_epoch.backward()
    optimizer.step()
    scheduler.step()
    # print(loss_in_epoch.item())
    train_loss += loss_in_epoch.item()
  print("Train Loss = ",train_loss/len(train_dataloader))

  model.eval()
  val_loss = 0
  with torch.no_grad():
    for batch in tqdm(test_dataloader):
      gen_qs, start_logits, end_logits, start_smax, end_smax, y_pre, y_en = model(batch['tokenized_input_qg'].to(device), batch['tokenized_input_position_qg'].to(device), batch['task_embedding_input_qg'].to(device), batch['segment_embedding_input_qg'].to(device), batch['question_input_ids'].to(device),
                                                                                batch['tokenized_input_qa'].to(device), batch['tokenized_input_position_qa'].to(device), batch['task_embedding_input_qa'].to(device), batch['segment_embedding_input_qa'].to(device),
                                                                                batch['tokenized_input_uw'].to(device), batch['tokenized_input_position_uw'].to(device), batch['task_embedding_input_uw'].to(device), batch['segment_embedding_input_uw'].to(device))
      target_qs = batch['question_input_ids'][:, 1:].to(device)
      qg_loss = criterion(gen_qs.view(-1, gen_qs.shape[-1]), target_qs.view(-1))

      answer_start_pos = batch['answer_token_start'].to(device)
      answer_end_pos = batch['answer_token_end'].to(device)
      qa_loss = criterion(start_logits, answer_start_pos) + criterion(end_logits, answer_end_pos)

      uw_loss = -torch.sum(y_en * torch.log(y_pre))

      loss_in_epoch = qg_loss + alpha * qa_loss + beta * uw_loss
      # print(loss_in_epoch.item())
      val_loss += loss_in_epoch.item()
  val_loss = val_loss/len(test_dataloader)
  print("Validation Loss = ",val_loss)

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save({
        "model_param": model.state_dict(),
        "optim_param": optimizer.state_dict(),
        "bst_dev_loss": best_val_loss,
        "epoch": epoch,
        "learning_rate": lr},
               f"/content/drive/MyDrive/NLPwithDeepLearning/FinalProject/QwithKD.pth")


Epoch 1/2


100%|██████████| 2500/2500 [15:32<00:00,  2.68it/s]


Train Loss =  59.56113160219193


100%|██████████| 250/250 [00:31<00:00,  7.86it/s]


Validation Loss =  5.163768609046936
Epoch 2/2


100%|██████████| 2500/2500 [15:39<00:00,  2.66it/s]


Train Loss =  5.000636830329895


100%|██████████| 250/250 [00:32<00:00,  7.81it/s]


Validation Loss =  4.397780145168304


#Generating Question

##Single Test sample

In [45]:
beam_size = 10
max_ques_len = 64

In [46]:
temp_dataset = test_dataset[9:10]
print(temp_dataset['question'])

['What does AFC stand for?']


In [48]:
def generate_question(model, temp_dataset, beam_size = 10, max_ques_len = 64):
  model.eval()
  with torch.no_grad():
    sequences = [[[tokenizer.bos_token_id], 0.0]]
    for m in range(max_ques_len):
      temp_sequences = []
      for i in range(len(sequences)):
        sequence = sequences[i][0]
        score = sequences[i][1]
        input_question_ids = [sequence + [tokenizer.eos_token_id]]
        ques, sl, el, ssm, esm, y_pre, y_en = model(torch.tensor(temp_dataset['tokenized_input_qg']).to(device), torch.tensor(temp_dataset['tokenized_input_position_qg']).to(device), torch.tensor(temp_dataset['task_embedding_input_qg']).to(device), torch.tensor(temp_dataset['segment_embedding_input_qg']).to(device), torch.tensor(input_question_ids).to(device),
                                                    torch.tensor(temp_dataset['tokenized_input_qa']).to(device), torch.tensor(temp_dataset['tokenized_input_position_qa']).to(device), torch.tensor(temp_dataset['task_embedding_input_qa']).to(device), torch.tensor(temp_dataset['segment_embedding_input_qa']).to(device),
                                                    torch.tensor(temp_dataset['tokenized_input_uw']).to(device), torch.tensor(temp_dataset['tokenized_input_position_uw']).to(device), torch.tensor(temp_dataset['task_embedding_input_uw']).to(device), torch.tensor(temp_dataset['segment_embedding_input_uw']).to(device))
        last_prediction = ques[0,-1]
        top_k_prob, top_k_ids = torch.topk(torch.softmax(last_prediction, dim = -1), beam_size)
        for j in range(beam_size):
          temp_seq = sequence + [top_k_ids[j].item()]
          temp_score = np.log(top_k_prob[j].item())
          temp_sequences.append([temp_seq,temp_score])
      temp_sequences = sorted(temp_sequences, key=lambda x: x[1], reverse=True)
      # print(temp_sequences)
      sequences = temp_sequences[:beam_size]

      results = [n[0][-1] == tokenizer.eos_token_id for n in sequences]
      if all(results):
        break;
    final_ids = sequences[0]
    # print(final_ids)
    qs = tokenizer.decode(final_ids[0], skip_special_tokens=True)
    bleu_score = corpus_bleu([qs], [temp_dataset['question']])

  return qs, bleu_score



In [49]:
gen_qs, bscore = generate_question(model, test_dataset[15:16])

In [50]:
print(test_dataset[15:16]['question'])
print(gen_qs)
print(bscore.score)

['Super Bowl 50 decided the NFL champion for what season?']
How year was Super Bowl 50 held?
15.181939159382823


##Taking random samples from the test set and generate for them

In [34]:
n = 10
min_value = 0
max_value = len(test_dataset)
random_indices = [random.randint(min_value, max_value) for _ in range(n)]
random_samples = test_dataset.select(random_indices)

In [35]:
model_path = f"/content/drive/MyDrive/NLPwithDeepLearning/FinalProject/QwithKD.pth"
checkpoint = torch.load(model_path)
model = QuesitonGenerationWithKnowledgeDist(embedding_dim= dimension_of_model, vocab_size = tokenizer.vocab_size, pad_token_id=tokenizer.pad_token_id)
model = model.to(device)

In [36]:
for i in range(len(random_samples)):
  gen_qs, bscore = generate_question(model, random_samples[i:i+1])
  print(random_samples[i:i+1]['question'])
  print(gen_qs)
  print(bscore.score)

['How many days did the Warsaw Uprising last?']
 hatchedBy July 1944, the Red Army was the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army, the Black Army
0.5749785925016119
['Which NFL team represented the NFC at Super Bowl 50?']
TheSuper Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) and The American Football League (AFC) and the American Football League (AFC) and the American Football League (AFC) and the American
1.4627051116799126
['What formation has an asymmetrical pattern of different terraces?']
NASAWarsaw is located on two main geomorphologic formations: the black and brown and the black and the brown and the brown and the brown and the brown and the brown and the brown and the brown and the brown and the brown

#Calculate BLEU Score for the test set

In [ ]:
total_bleu = 0
for i in range(len(test_dataset)):
  gen_qs, bscore = generate_question(model, test_dataset[i:i+1])
  total_bleu += bscore.score
print("BLEU score of the test set = ",total_bleu/len(test_dataset))

END. Ignore the rest

In [ ]:
from sacrebleu import corpus_bleu

bleu_score = corpus_bleu([qs], [temp_dataset['question']])
print("BLEU Score:", bleu_score.score)

BLEU Score: 3.7477767366779213


In [ ]:
seq = [[[0, 2264], -0.783766805579641], [[0, 12375], -2.4410916260344084], [[0, 6179], -2.45050151204054], [[0, 1779], -2.650597617040899], [[0, 32251], -2.819804111664899], [[0, 1121], -2.9759483646288953], [[0, 133], -3.7028188747980817], [[0, 13841], -3.8143216006245253], [[0, 7608], -4.703337684861119], [[0, 4148], -4.849275677527714]]

# Sort the sequences based on the second value in each sub-list
sorted_sequences = sorted(seq, key=lambda x: x[1], reverse=True)
top_5_sequences = sorted_sequences[:5]
print(top_5_sequences)

[[[0, 2264], -0.783766805579641], [[0, 12375], -2.4410916260344084], [[0, 6179], -2.45050151204054], [[0, 1779], -2.650597617040899], [[0, 32251], -2.819804111664899]]


In [ ]:
results = [sequence[0][-1] == 2264 for sequence in top_5_sequences]
print(results)

[True, False, False, False, False]


In [ ]:
r = [True, True, True, True, True]

In [ ]:
if all(results):
    print("All values in 'results' are True.")
else:
    print("Not all values in 'results' are True.")

All values in 'results' are True.


In [ ]:
temp_dataset = test_dataset[5:6]
print(temp_dataset)

{'id': ['56be8e613aeaaa14008c90d1'], 'title': ['Super_Bowl_50'], 'context': ['Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.'], 'question': ['What was the theme of Super Bowl 50?'], 'answers': [{'text': ['"golden anniversary"', 'gold-themed', '"golden anniversary'], 'an

In [ ]:
print(temp_dataset['question_input_ids'][0])

[0, 2264, 21, 5, 4782, 9, 1582, 2616, 654, 116, 2]


In [ ]:
print(tokenizer.decode(torch.tensor(temp_dataset['question_input_ids'][0]), skip_special_tokens=True))

What was the theme of Super Bowl 50?


In [ ]:
ques, sl, el, ssm, esm, y_pre, y_en = model(torch.tensor(temp_dataset['tokenized_input_qg']).to(device), torch.tensor(temp_dataset['tokenized_input_position_qg']).to(device), torch.tensor(temp_dataset['task_embedding_input_qg']).to(device), torch.tensor(temp_dataset['segment_embedding_input_qg']).to(device), torch.tensor(temp_dataset['question_input_ids']).to(device),
                                            torch.tensor(temp_dataset['tokenized_input_qa']).to(device), torch.tensor(temp_dataset['tokenized_input_position_qa']).to(device), torch.tensor(temp_dataset['task_embedding_input_qa']).to(device), torch.tensor(temp_dataset['segment_embedding_input_qa']).to(device),
                                            torch.tensor(temp_dataset['tokenized_input_uw']).to(device), torch.tensor(temp_dataset['tokenized_input_position_uw']).to(device), torch.tensor(temp_dataset['task_embedding_input_uw']).to(device), torch.tensor(temp_dataset['segment_embedding_input_uw']).to(device))

In [ ]:
hypotheses = [([tokenizer.bos_token_id], 0.0)]

In [ ]:
hypothesis,score = hypotheses[0]
print(hypothesis)
print(score)

[0]
0.0


In [ ]:
input_hypothesis = torch.tensor(hypothesis + [tokenizer.eos_token_id]).unsqueeze(0).to(device)

In [ ]:
input_hypothesis2 = torch.tensor([hypothesis + [tokenizer.eos_token_id]])

In [ ]:
print(input_hypothesis2)

tensor([[0, 2]])


In [ ]:
ques, sl, el, ssm, esm, y_pre, y_en = model(torch.tensor(temp_dataset['tokenized_input_qg']).to(device), torch.tensor(temp_dataset['tokenized_input_position_qg']).to(device), torch.tensor(temp_dataset['task_embedding_input_qg']).to(device), torch.tensor(temp_dataset['segment_embedding_input_qg']).to(device), input_hypothesis,
                                            torch.tensor(temp_dataset['tokenized_input_qa']).to(device), torch.tensor(temp_dataset['tokenized_input_position_qa']).to(device), torch.tensor(temp_dataset['task_embedding_input_qa']).to(device), torch.tensor(temp_dataset['segment_embedding_input_qa']).to(device),
                                            torch.tensor(temp_dataset['tokenized_input_uw']).to(device), torch.tensor(temp_dataset['tokenized_input_position_uw']).to(device), torch.tensor(temp_dataset['task_embedding_input_uw']).to(device), torch.tensor(temp_dataset['segment_embedding_input_uw']).to(device))

In [ ]:
print(ques.shape)

torch.Size([1, 2, 50265])


In [ ]:
print(ques)

tensor([[[31.3849,  2.7021, 11.8357,  ..., -0.1430, -0.4179, -0.6786],
         [ 0.8995, -0.4602,  5.4628,  ..., -1.8235, -2.0595, -2.3240]]],
       device='cuda:0')


In [ ]:
last_ques = ques[0, -1]
print(last_ques.shape)

torch.Size([50265])


In [ ]:
print(last_ques)

tensor([ 0.8995, -0.4602,  5.4628,  ..., -1.8235, -2.0595, -2.3240],
       device='cuda:0', grad_fn=<SelectBackward0>)


In [ ]:
top_k_prob, top_k_ids = torch.topk(torch.softmax(last_ques, dim = -1), 10)

In [ ]:
print(top_k_prob.shape, top_k_ids.shape)

torch.Size([10]) torch.Size([10])


In [ ]:
print(top_k_prob)

tensor([0.4567, 0.0871, 0.0863, 0.0706, 0.0596, 0.0510, 0.0247, 0.0221, 0.0091,
        0.0078], device='cuda:0', grad_fn=<TopkBackward0>)


In [ ]:
print(torch.log(top_k_prob[1]).item())

-2.441091537475586


In [ ]:
import numpy as np
np.log(top_k_prob[1].item())

-2.4410916260344084

In [ ]:
temp_dataset = train_dataset[0:1]

In [ ]:
dimension_of_model = AutoModel.from_pretrained(model_name).shared.weight.shape[1]
print(dimension_of_model)

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

768


In [ ]:
encoder = QuesitonGenerationWithKnowledgeDist(embedding_dim= dimension_of_model, vocab_size = tokenizer.vocab_size, pad_token_id=tokenizer.pad_token_id)

In [ ]:
ques, sl, el, ssm, esm, y_pre, y_en = encoder(torch.tensor(temp_dataset['tokenized_input_qg']), torch.tensor(temp_dataset['tokenized_input_position_qg']), torch.tensor(temp_dataset['task_embedding_input_qg']), torch.tensor(temp_dataset['segment_embedding_input_qg']), torch.tensor(temp_dataset['question_input_ids']),
                 torch.tensor(temp_dataset['tokenized_input_qa']), torch.tensor(temp_dataset['tokenized_input_position_qa']), torch.tensor(temp_dataset['task_embedding_input_qa']), torch.tensor(temp_dataset['segment_embedding_input_qa']),
                 torch.tensor(temp_dataset['tokenized_input_uw']), torch.tensor(temp_dataset['tokenized_input_position_uw']), torch.tensor(temp_dataset['task_embedding_input_uw']), torch.tensor(temp_dataset['segment_embedding_input_uw']))

In [ ]:
print(ques.shape)

torch.Size([1, 19, 50265])


In [ ]:
target_ids = torch.tensor(temp_dataset['question_input_ids'])
print(target_ids.shape)

torch.Size([1, 19])


In [ ]:
print(ques.shape[-1])

50265


In [ ]:
qg_loss = criterion(ques.view(-1, ques.shape[-1]), target_ids.view(-1))
print(qg_loss)

tensor(7.8833, grad_fn=<NllLossBackward0>)


In [ ]:
answer_start = torch.tensor(temp_dataset['answer_token_start'])
answer_end = torch.tensor(temp_dataset['answer_token_end'])

In [ ]:
print(answer_start)
print(answer_end)

tensor([117])
tensor([124])


In [ ]:
print(sl.shape)
print(el.shape)

torch.Size([1, 181])
torch.Size([1, 181])


In [ ]:
qa_loss = criterion(sl, answer_start) + criterion(el, answer_end)
print(qa_loss)

tensor(10.3349, grad_fn=<AddBackward0>)


In [ ]:
kd_loss = -torch.sum(y_en * torch.log(y_pre))
print(kd_loss)

tensor(179.3923, grad_fn=<NegBackward0>)


In [ ]:
total_loss = qg_loss + 0.8 * qa_loss + 0.15 * kd_loss
print(total_loss)

tensor(43.0601, grad_fn=<AddBackward0>)


In [ ]:
primal_dual_embeddings = torch.randint(0, 10, size=(3, 5, 4))
print(primal_dual_embeddings)

tensor([[[5, 7, 6, 1],
         [3, 6, 7, 3],
         [5, 9, 5, 0],
         [6, 3, 0, 5],
         [4, 8, 0, 9]],

        [[4, 2, 1, 9],
         [3, 4, 1, 6],
         [6, 7, 3, 3],
         [8, 3, 5, 0],
         [9, 1, 0, 2]],

        [[0, 7, 3, 1],
         [0, 8, 3, 4],
         [3, 5, 5, 4],
         [1, 7, 5, 2],
         [3, 7, 7, 8]]])


In [ ]:
random_values = torch.rand(3, 5)

# Convert random values to boolean (True or False)
input_mask = random_values < 0.30

print(input_mask)

tensor([[False, False, False,  True,  True],
        [ True, False, False, False,  True],
        [ True,  True,  True,  True, False]])


In [ ]:
masked_word_embeddings_en = primal_dual_embeddings.masked_select(input_mask.unsqueeze(-1)).view(-1, primal_dual_embeddings.shape[-1])
print(masked_word_embeddings_en)

tensor([[6, 3, 0, 5],
        [4, 8, 0, 9],
        [4, 2, 1, 9],
        [9, 1, 0, 2],
        [0, 7, 3, 1],
        [0, 8, 3, 4],
        [3, 5, 5, 4],
        [1, 7, 5, 2]])


In [ ]:
a = primal_dual_embeddings
b = input_mask

In [ ]:
print(a.masked_select(b.unsqueeze(-1)))

tensor([6, 3, 0, 5, 4, 8, 0, 9, 4, 2, 1, 9, 9, 1, 0, 2, 0, 7, 3, 1, 0, 8, 3, 4,
        3, 5, 5, 4, 1, 7, 5, 2])


In [ ]:
print(a[b.unsqueeze(-1)])

IndexError: ignored

In [ ]:
print(a.masked_select(b.unsqueeze(-1)).reshape(-1,a.shape[1]))

RuntimeError: ignored

In [ ]:
print(a.shape[-1])

4
